In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import copy
import torch
from sklearn.metrics import f1_score
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [ ]:
# load vectors
train_vectors_df = pd.read_csv("./../data/train_pca.csv")
val_vectors_df = pd.read_csv("./../data/val_pca.csv")
test_vectors_df = pd.read_csv("./../data/test_pca.csv")

# load features with labels
train_features_df = pd.read_csv("./../data/train_features.csv")
val_features_df = pd.read_csv("./../data/val_features.csv")
test_features_df = pd.read_csv("./../data/test_features.csv")

# dataset

In [ ]:
class FusionDataset(Dataset):

    def __init__(self, vectors_df, features_df, label_col="label"):
        self.vectors = torch.tensor(vectors_df.values, dtype=torch.float32)

        self.features = torch.tensor(
            features_df.drop(columns=[label_col]).values,
            dtype=torch.float32
        )

        self.labels = torch.tensor(
            features_df[label_col].values,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.vectors[idx], self.features[idx], self.labels[idx]

In [ ]:
train_dataset = FusionDataset(train_vectors_df, train_features_df)
val_dataset = FusionDataset(val_vectors_df, val_features_df)
test_dataset = FusionDataset(test_vectors_df, test_features_df)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=4)

# model

In [ ]:
class FusionNet(nn.Module):
    def __init__(self, vector_dim=360, feature_dim=20, num_classes=2, dropout=0.3):
        super().__init__()

        self.vector_branch = nn.Sequential(
            nn.Linear(vector_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU()
        )

        self.feature_branch = nn.Sequential(
            nn.Linear(feature_dim, 32),
            nn.BatchNorm1d(32),
            nn.ReLU()
        )

        self.classifier = nn.Sequential(
            nn.Linear(160, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )

    def forward(self, vectors, features):
        v = self.vector_branch(vectors)
        f = self.feature_branch(features)
        x = torch.cat([v, f], dim=1)

        return self.classifier(x)

# overfit one batch

In [ ]:
def overfit_one_batch(model, train_loader, device, epochs=500):
    model.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
    
    vectors, features, labels = next(iter(train_loader))

    vectors = vectors.to(device).float()
    features = features.to(device).float()
    labels = labels.to(device).long()

    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(vectors, features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        preds = outputs.argmax(dim=1)
        acc = (preds == labels).float().mean().item()

        if epoch % 20 == 0 or acc == 1.0:
            print(f"Epoch {epoch:4d} | Loss: {loss.item():.6f} | Acc: {acc:.4f}")

        if acc == 1.0:
            print("Successfully overfit one batch!")
            break

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FusionNet(vector_dim=360, feature_dim=20, num_classes=2, dropout=0.0).to(device)
overfit_one_batch(model, train_loader, device, epochs=500)

# train loop

In [ ]:
def train_model(model, train_loader, val_loader, device, epochs=50, patience=7, model_path="best_model.pth"):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
    scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
    best_f1 = 0.0
    best_weights = None
    wait = 0

    for epoch in range(epochs):

        model.train()
        train_loss = 0.0

        for vectors, features, labels in train_loader:
            vectors = vectors.to(device).float()
            features = features.to(device).float()
            labels = labels.to(device).long()
            optimizer.zero_grad()

            outputs = model(vectors, features)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0.0
        preds = []
        targets = []

        with torch.no_grad():
            for vectors, features, labels in val_loader:
                vectors = vectors.to(device).float()
                features = features.to(device).float()
                labels = labels.to(device).long()

                outputs = model(vectors, features)
                loss = criterion(outputs, labels)

                val_loss += loss.item()

                pred = outputs.argmax(dim=1)

                preds.extend(pred.cpu().numpy())
                targets.extend(labels.cpu().numpy())

        val_loss /= len(val_loader)
        val_f1 = f1_score(targets, preds, average="macro")

        scheduler.step(val_f1)

        current_lr = optimizer.param_groups[0]["lr"]

        print(f"Epoch {epoch + 1:3d} | Train Loss {train_loss:.4f} | Val Loss {val_loss:.4f} | Val F1 {val_f1:.4f} | LR {current_lr:.6f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            wait = 0
            best_weights = copy.deepcopy(model.state_dict())
            torch.save(best_weights, model_path)
            print("Validation F1 improved -> model saved.")
        else:
            wait += 1
            print(f"No improvement ({wait}/{patience})")

            if wait >= patience:
                print("Early stopping.")
                break

    if best_weights is not None:
        model.load_state_dict(best_weights)

    return model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FusionNet(vector_dim=360, feature_dim=20, num_classes=2, dropout=0.3).to(device)
model = train_model(model, train_loader, val_loader, device, epochs=50, patience=7, model_path="best_fusion_model.pth")